# 드럼 분리 입력 채보 모델 비교 — GPU 서버 실행 (S15P31A307-164, 설계 CHART-003-real-drum-data-check 작업 10)

서비스와 같은 입력(노래에서 BS-Roformer SW로 뽑은 드럼)으로 채보 모델 후보를 비교한다. 이 노트북은 예측만 만들고, 채점과 표는 로컬 `scripts/score_sep_compare.py`가 한다.

| 이름 | 무엇 | 이 노트북에서 돌리는 것 |
| --- | --- | --- |
| 현재 사용 중 모델 | 원래 ADTOF + 드럼 킷 분리. 11/02 당시 서비스 채보 모델 | `adtof_drumsep.yaml`로 MDB 23곡·ENST 64곡 |
| 실제 녹음 학습 모델 | ENST·MDB를 더해 학습한 모델. 전체로 학습한 `tom_6cls_real_full`은 되돌리기용 combo 모델에만 남음 | 학습에 쓴 곡은 채점하면 안 되므로, 드러머·곡 묶음을 하나씩 빼고 학습한 버전 3개(`realrec_heldout_d1~3`)로 각자 안 본 곡만 |
| 조합: 탐만 현재 모델 | 탐 3종은 현재 사용 중 모델, 나머지는 실제 녹음 학습 모델 | 따로 안 돌림. 로컬에서 위 두 예측을 합쳐 채점 |

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에, 이 파일을 `ssaplash/notebooks/`에 올린다. 서버에는 `.venv`, 데이터 링크(`data/enst`, `data/mdb_drums`), 가중치 `data/models/adtof_ft/tom_6cls_real_no_d{1,2,3}`이 이미 있다.

**순서.** 1 설정 → 1-1 지금 상태 → 2 코드 덮어 풀기 → 3 분리 음원·가중치 확인 → 4 예측 시작(run 8개 동시, 백그라운드) → 5 진행 확인(여러 번) → 6 결과 묶기. 끊기면 1번 → 4번을 치면 도는 run은 그대로 두고 멈춘 run만 이어 간다. 5번으로 확인한다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import getpass
import os
import sys

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
ME = getpass.getuser()
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
RES = "data/results/sep_compare"          # run 폴더들이 여기 생김
DS = "configs/bench/datasets"
TR = "configs/bench/transcribers/adtof_drumsep"
# 드럼을 미리 뽑아 둔 폴더(곡 폴더 아래 drums.wav). 11/02 기준 서버에 없어서 None. None이면 4번이 BS-Roformer SW로 새로 뽑음
SEP_MDB = None
SEP_ENST = None
SEPARATOR = "configs/bench/models/msst/bs_roformer_sw.yaml"
# run 이름 → (채보 설정, 데이터셋, 분리 음원 폴더, 기대 곡 수)
RUNS = {
    "current_mdb": (f"{TR}/adtof_drumsep.yaml", f"{DS}/mdb_drums.yaml", "mdb", 23),
    "current_enst": (f"{TR}/adtof_drumsep.yaml", f"{DS}/enst_minus_one.yaml", "enst", 64),
}
for k, (nm, ne) in zip((1, 2, 3), ((8, 21), (7, 22), (8, 21)), strict=True):
    RUNS[f"realrec_mdb_d{k}"] = (f"{TR}/sep_compare/realrec_heldout_d{k}.yaml", f"{DS}/real_check/mdb_fold{k}.yaml", "mdb", nm)
    RUNS[f"realrec_enst_d{k}"] = (f"{TR}/sep_compare/realrec_heldout_d{k}.yaml", f"{DS}/real_check/enst_minus_one_d{k}.yaml", "enst", ne)
os.makedirs(RESULTS, exist_ok=True)
%cd $AI
print("run", len(RUNS), "개:", ", ".join(RUNS))
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!ls -la $UPLOAD/S15P31A307_code.zip
# 확인: 'run 8 개: current_mdb, current_enst, realrec_mdb_d1, realrec_enst_d1, …', GPU 한 줄(H200).
#       memory.used가 100,000MiB를 넘으면 다른 사용자가 크게 쓰는 중이니 기다림. 코드 zip 날짜가 오늘 올린 것

In [ ]:
# 1-1. 지금 상태 (여러 번 쳐도 됨)
!ps -u $ME -o pid,etime,pcpu,args | grep -E "[r]un-transcription|[a]dtof-finetune|[r]ule-features" | cut -c1-160
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
# 확인: 4번을 치기 전이면 ps 줄이 없음. 4번 뒤에는 run-transcription 한 줄이 돎

In [ ]:
# 2. 코드 덮어 풀기. 기존 폴더를 지우지 않고 위에 덮어씀 (.venv·데이터 링크·결과가 그대로 남음)
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!ls $TR/sep_compare/ scripts/score_sep_compare.py
!grep -c '"HIHAT_PEDAL": 44' src/ssaplash_ai/bench/transcribers/transcriber_base.py
# 확인: realrec_heldout_d1·d2·d3.yaml 3개와 score_sep_compare.py, 숫자 1.
#       숫자가 0이면 zip이 예전 것(예측 MIDI에서 하이햇 페달이 킥으로 저장됨). 로컬에서 zip을 다시 만들어 올림

In [ ]:
# 3. 분리 음원·가중치 확인
import glob

for key, path in (("mdb", SEP_MDB), ("enst", SEP_ENST)):
    n = len(glob.glob(f"{path}/**/drums.wav", recursive=True)) if path and os.path.isdir(path) else 0
    print(key, path, "drums.wav", n)
!ls -la data/models/adtof_ft/tom_6cls_real_no_d1/best.pth data/models/adtof_ft/tom_6cls_real_no_d2/best.pth data/models/adtof_ft/tom_6cls_real_no_d3/best.pth
!ls data/models/mdx23c_drumsep/ | head -3
# 확인: 'mdb None drums.wav 0', 'enst None drums.wav 0'(드럼을 4번에서 새로 뽑으므로 정상), best.pth 3줄,
#   mdx23c_drumsep 가중치 파일 두 줄. best.pth가 없으면 그 줄을 적어 두고 멈춤

In [ ]:
# 4. 예측 시작 (GPU, 백그라운드, 한 번만). run 8개를 동시에 띄움. 다시 치면 도는 run은 건너뛰고, 멈춘 run은 끝난 곡을 건너뛰고 이어 감
#    run마다 드럼 분리(SEP_*가 None일 때)·채보 모델을 따로 올려 GPU 메모리를 run당 2~5GB 씀. 모두 합쳐 20~40분
import subprocess

SEP = {"mdb": SEP_MDB, "enst": SEP_ENST}
RUN_LOG = {run: f"{AI}/sep_compare_{run}.log" for run in RUNS}


def alive(run):
    try:
        pid = int(open(RUN_LOG[run] + ".pid").read())
        with open(f"/proc/{pid}/stat") as f:
            return f.read().rsplit(")", 1)[1].split()[0] != "Z"
    except (OSError, ValueError, IndexError):
        return False


for run, (model, ds, key, _) in RUNS.items():
    if alive(run):
        print(run, "이미 도는 중, 건너뜀")
        continue
    src = f"--separated-dir {SEP[key]}" if SEP[key] else f"--separator {SEPARATOR}"
    cmd = (f".venv/bin/ssaplash-bench run-transcription --models {model} --dataset {ds} --input separated {src} "
           f"--out {RES} --run-id {run} --save-midi 9999 --device cuda --mlflow-uri none")
    proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", cmd], cwd=AI, stdout=open(RUN_LOG[run], "a"),
                            stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
    with open(RUN_LOG[run] + ".pid", "w") as f:
        f.write(str(proc.pid))
    print("시작.", run, "pid", proc.pid)
# 확인: '시작. current_mdb pid …'부터 8줄. 다시 치면 도는 run은 '이미 도는 중, 건너뜀'
#   GPU 메모리가 모자라 'CUDA out of memory'가 5번 로그에 나오면, 그 run만 끝난 뒤 이 셀을 다시 침(빈 run만 다시 뜸)

In [ ]:
# 5. 진행 확인 (여러 번)
for run, (_, _, _, n) in RUNS.items():
    got = len(glob.glob(f"{RES}/{run}/midi/**/*.mid", recursive=True))
    state = "끝" if got >= n else ("도는 중" if alive(run) else "멈춤")
    print(f"{run:18s} 예측 MIDI {got:3d} / {n}  {state}")
!nvidia-smi -i $GPU_ID --query-gpu=memory.used,utilization.gpu --format=csv,noheader
!grep -l -E "Traceback|out of memory" {AI}/sep_compare_*.log 2>/dev/null || true
# 확인: 8줄이 모두 '끝'이면 6번. 처음 몇 분은 드럼 분리 중이라 숫자가 0에 머묾.
#   '멈춤'인데 '끝'이 아니면 아래 grep에 그 run 로그 이름이 나옴. `!tail -30 <로그>` 결과를 붙여 주거나, 4번을 다시 쳐서 이어 감
#   마지막 grep에 아무것도 안 나와야 정상

In [ ]:
# 6. 결과 묶기. results 폴더의 sep_compare_results.zip 을 내려받아 로컬 AI/data/results/ 에 둠 (푸는 것은 세션에서)
#    zip 안 경로는 AI/ 기준 상대 경로 그대로. 분리 음원(wav)은 빼고 예측 MIDI·결과표·로그만 담음
import zipfile

missing = []
with zipfile.ZipFile(f"{RESULTS}/sep_compare_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for run in RUNS:
        files = [p for p in glob.glob(f"{RES}/{run}/**/*", recursive=True)
                 if os.path.isfile(p) and not p.endswith(".wav")]
        if not files:
            missing.append(run)
        for p in files:
            z.write(p)
    for run in RUNS:
        if os.path.exists(RUN_LOG[run]):
            z.write(RUN_LOG[run], os.path.basename(RUN_LOG[run]))
!ls -la $RESULTS/sep_compare_results.zip
print("zip에 빠진 run:", missing or "없음")
# 확인: 'zip에 빠진 run: 없음'과 수 MB짜리 zip 한 줄. 빠진 run이 있으면 5번에서 '끝'이 아닌 것이니 기다렸다가 다시 6

## 7. 크래시·라이드를 직접 나누는 7클래스 모델 추가 (11/02)

현재 사용 중 모델(원래 ADTOF + 드럼 킷 분리)은 크래시·라이드를 드럼 킷 분리 소리 크기로 고른다. 크래시·라이드를 모델이 직접 나눠 내게 합성 데이터로 학습한 7클래스 모델(`tom_7cls`, 어댑터 `adtof_drumsep_ft7.yaml`)을 같은 입력으로 돌려, 심벌만 이 모델에서 가져오는 조합을 비교한다. 이 모델은 실제 녹음을 학습에 안 넣어서 MDB·ENST 모든 곡을 그대로 채점할 수 있다.

**순서.** 1번 → 7-1 시작(run 2개 동시) → 7-2 확인(여러 번, 약 20분) → 7-3 결과 묶기. 1~6번을 다시 칠 필요 없음.

In [ ]:
# 7-1. 7클래스 모델 run 2개 동시 시작 (GPU, 백그라운드, 한 번만. 다시 치면 도는 run은 건너뜀)
import glob
import subprocess

RUNS7 = {
    "cym7_mdb": (f"{TR}/adtof_drumsep_ft7.yaml", f"{DS}/mdb_drums.yaml", 23),
    "cym7_enst": (f"{TR}/adtof_drumsep_ft7.yaml", f"{DS}/enst_minus_one.yaml", 64),
}
LOG7 = {run: f"{AI}/sep_compare_{run}.log" for run in RUNS7}


def alive7(run):
    try:
        pid = int(open(LOG7[run] + ".pid").read())
        with open(f"/proc/{pid}/stat") as f:
            return f.read().rsplit(")", 1)[1].split()[0] != "Z"
    except (OSError, ValueError, IndexError):
        return False


!ls -la data/models/adtof_ft/tom_7cls/best.pth
for run, (model, ds, _) in RUNS7.items():
    if alive7(run):
        print(run, "이미 도는 중, 건너뜀")
        continue
    cmd = (f".venv/bin/ssaplash-bench run-transcription --models {model} --dataset {ds} --input separated "
           f"--separator {SEPARATOR} --out {RES} --run-id {run} --save-midi 9999 --device cuda --mlflow-uri none")
    proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", cmd], cwd=AI, stdout=open(LOG7[run], "a"),
                            stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
    with open(LOG7[run] + ".pid", "w") as f:
        f.write(str(proc.pid))
    print("시작.", run, "pid", proc.pid)
# 확인: best.pth 한 줄, '시작. cym7_mdb pid …', '시작. cym7_enst pid …'.
#   best.pth가 'No such file'이면 7클래스 가중치가 서버에서 지워진 것이니 그 줄을 붙여 주고 멈춤

In [ ]:
# 7-2. 진행 확인 (여러 번)
for run, (_, _, n) in RUNS7.items():
    got = len(glob.glob(f"{RES}/{run}/midi/**/*.mid", recursive=True))
    state = "끝" if got >= n else ("도는 중" if alive7(run) else "멈춤")
    print(f"{run:10s} 예측 MIDI {got:3d} / {n}  {state}")
!grep -l -E "Traceback|out of memory" {AI}/sep_compare_cym7_*.log 2>/dev/null || true
# 확인: 두 줄 모두 '끝'이면 7-3. 처음 몇 분은 드럼 분리 중이라 0에 머묾. 마지막 grep에 아무것도 안 나와야 정상.
#   '멈춤'이면 7-1을 다시 쳐서 이어 감. 로그 이름이 나오면 `!tail -30 <로그>` 결과를 붙여 줌

In [ ]:
# 7-3. 결과 묶기. results 폴더의 sep_compare_cym7.zip 을 내려받아 로컬 AI/data/results/ 에 둠
import zipfile

with zipfile.ZipFile(f"{RESULTS}/sep_compare_cym7.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for run in RUNS7:
        for p in glob.glob(f"{RES}/{run}/**/*", recursive=True):
            if os.path.isfile(p) and not p.endswith(".wav"):
                z.write(p)
        if os.path.exists(LOG7[run]):
            z.write(LOG7[run], os.path.basename(LOG7[run]))
!ls -la $RESULTS/sep_compare_cym7.zip
print("예측 MIDI", sum(n.endswith(".mid") for n in zipfile.ZipFile(f"{RESULTS}/sep_compare_cym7.zip").namelist()), "개")
# 확인: 수백 KB짜리 zip 한 줄과 '예측 MIDI 87 개'(MDB 23 + ENST 64). 숫자가 작으면 7-2에서 '끝'이 아닌 run이 있는 것